# Modeling

This notebook defines the modeling data, chronological evaluation periods, baseline benchmarks, and first learned models for attendance prediction.

## 1. Load the Feature Dataset

The dataset contains one row per prediction instance and all feature groups constructed in Notebook 03. It is loaded without recalculating any engineered features.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from src.modeling import (
    PREDICTION_IDENTIFIER_COLUMNS,
    assign_temporal_split,
    validate_prediction_identity,
    validate_prediction_time,
)

FEATURE_DATASET_PATH = (
    Path("..")
    / "data"
    / "processed"
    / "baseline_plus_historical_plus_member_plus_reliability_plus_dynamics_plus_context_features.parquet"
)

features = pd.read_parquet(FEATURE_DATASET_PATH)
features.shape

## 2. Define Target and Feature Groups

The target is the final number of attendees. The predictors remain grouped by their analytical origin so that later model comparisons can evaluate the contribution of baseline, historical, member-level, booking-dynamics, and class-context information. `studio` and `course` are identifier components as well as categorical predictors.

In [ ]:
target_column = "final_attendance_count"

physical_class_identifier_columns = [
    "studio",
    "course",
    "class_start",
]

prediction_identifier_columns = PREDICTION_IDENTIFIER_COLUMNS.copy()

baseline_features = [
    "studio",
    "course",
    "instructor",
    "weekday",
    "is_holiday_week",
    "is_full",
    "has_waiting_list",
    "capacity",
    "waiting_list_length",
    "snapshot_age_hours",
    "attendance_count",
    "available_spots",
    "occupancy_rate",
    "class_hour",
]

historical_features = [
    f"{prefix}_attendance_{stat}_{window}"
    for prefix in ["course", "instructor", "studio"]
    for window in ["30d", "90d", "365d"]
    for stat in ["mean", "std", "count"]
]

member_history_features = [
    f"{feature}_{window}"
    for feature in [
        "member_attendance_count_mean",
        "members_with_history_count",
        "members_with_history_share",
        "member_course_affinity_mean",
        "member_instructor_affinity_mean",
    ]
    for window in ["30d", "90d", "365d"]
]

member_reliability_features = [
    f"{feature}_{window}"
    for feature in [
        "member_booked_at_horizon_count_mean",
        "members_with_reliability_history_count",
        "members_with_reliability_history_share",
        "member_show_up_rate_mean",
    ]
    for window in ["30d", "90d", "365d"]
]

booking_dynamics_features = [
    f"{feature}_{window}"
    for feature in [
        "observed_booking_increase_events",
        "observed_booking_decrease_events",
        "observed_net_booking_change",
    ]
    for window in ["6h", "24h", "72h"]
] + [
    "max_observed_attendance_before_prediction",
    "attendance_drop_from_observed_peak",
]

class_context_features = [
    "total_demand_count",
    "total_demand_fill_ratio",
    "course_instructor_attendance_mean_90d",
    "course_instructor_history_count_90d",
]

feature_groups = {
    "baseline": baseline_features,
    "historical": historical_features,
    "member_history": member_history_features,
    "member_reliability": member_reliability_features,
    "booking_dynamics": booking_dynamics_features,
    "class_context": class_context_features,
}

all_model_features = [
    feature
    for group_features in feature_groups.values()
    for feature in group_features
]

In [ ]:
feature_group_summary = pd.Series(
    {group: len(columns) for group, columns in feature_groups.items()},
    name="number_of_features",
).rename_axis("feature_group")

if target_column not in features.columns:
    raise ValueError(f"Missing required target column: {target_column!r}.")
missing_feature_columns = sorted(set(all_model_features) - set(features.columns))
if missing_feature_columns:
    raise ValueError(f"Missing model features: {missing_feature_columns}")
if len(all_model_features) != len(set(all_model_features)):
    raise ValueError("Model feature names must be unique.")
if target_column in all_model_features:
    raise ValueError("The target must not be included in the model features.")

feature_group_summary.to_frame()

## 3. Validate Core Modeling Assumptions

Before modeling, we verify the two structural assumptions established in the preceding notebooks: every prediction instance is unique, and each `prediction_time` corresponds to its defined horizon before `class_start`.

In [ ]:
validate_prediction_identity(features)
validate_prediction_time(features)

## 4. Chronological Train / Validation / Test Split

Attendance prediction is temporal: historical features contain only information available at each prediction time. A random split could train the model on classes occurring later than some validation or test classes. A chronological split therefore better represents the intended deployment setting: training on the past and predicting future classes. The split is based solely on `class_start`, so all prediction horizons for the same physical class remain in the same partition.

In [ ]:
class_start_timezone = features["class_start"].dt.tz
training_end = pd.Timestamp("2026-02-01", tz=class_start_timezone)
validation_end = pd.Timestamp("2026-05-01", tz=class_start_timezone)

features = features.copy()
features["split"] = assign_temporal_split(
    features,
    training_end=training_end,
    validation_end=validation_end,
)

## 5. Inspect the Split

The split summary describes the amount of data, temporal coverage, and target distribution available in each evaluation period. Physical-class counts treat all prediction horizons of the same class as one class.

In [ ]:
split_order = ["train", "validation", "test"]
physical_class_counts = (
    features[physical_class_identifier_columns + ["split"]]
    .drop_duplicates()
    .groupby("split")
    .size()
)

split_summary = (
    features.groupby("split")
    .agg(
        prediction_instances=("split", "size"),
        earliest_class_start=("class_start", "min"),
        latest_class_start=("class_start", "max"),
        mean_final_attendance=(target_column, "mean"),
        std_final_attendance=(target_column, "std"),
    )
    .reindex(split_order)
)
split_summary.insert(1, "physical_classes", physical_class_counts)
split_summary.assign(
    mean_final_attendance=split_summary["mean_final_attendance"].round(2),
    std_final_attendance=split_summary["std_final_attendance"].round(2),
)

Training contains the earliest classes, validation follows chronologically, and test represents the most recent period. On the original dataset, mean attendance ranges from 5.34 to 5.70 and its standard deviation from 2.57 to 2.67, indicating no large shift in the target distribution across the three periods.

## 6. Prepare for the First Model

The supervised-learning inputs are now defined. We first establish simple prediction benchmarks before fitting the first machine-learning model.

In [ ]:
train = features.loc[features["split"] == "train"]
validation = features.loc[features["split"] == "validation"]

X_train, y_train = train[all_model_features], train[target_column]
X_validation, y_validation = (
    validation[all_model_features],
    validation[target_column],
)

## 7. Baseline Prediction Benchmarks

Before fitting a machine-learning model, two simple validation benchmarks establish the performance that a useful model should improve upon. The training-mean baseline ignores all class information, while the current-booking baseline uses the number of members booked at prediction time. The test set remains deliberately untouched until the final model has been selected.

In [ ]:
current_booking_feature = "attendance_count"

if train.empty or validation.empty:
    raise ValueError("Training and validation sets must not be empty.")
if y_train.isna().any() or y_validation.isna().any():
    raise ValueError("Training and validation targets must not be missing.")
if current_booking_feature not in validation.columns:
    raise ValueError(f"Missing benchmark feature: {current_booking_feature!r}.")

training_mean = y_train.mean()
training_mean_prediction = pd.Series(training_mean, index=y_validation.index)
current_booking_prediction = validation[current_booking_feature]

def evaluate_predictions(actual, prediction):
    prediction = pd.Series(prediction, index=actual.index)
    return {
        "MAE": mean_absolute_error(actual, prediction),
        "RMSE": root_mean_squared_error(actual, prediction),
        "mean_error": (prediction - actual).mean(),
    }

benchmark_results = pd.DataFrame(
    [
        {
            "model": "Training mean",
            **evaluate_predictions(y_validation, training_mean_prediction),
        },
        {
            "model": "Current booking count",
            **evaluate_predictions(y_validation, current_booking_prediction),
        },
    ]
).set_index("model")

benchmark_results.round(2)

On the original dataset, the current booking count improves on the training-mean benchmark for both MAE and RMSE, although the gain is modest. Its positive mean error indicates a slight tendency to overpredict final attendance. An MAE close to two attendees suggests that cancellations, no-shows, and bookings after prediction time still leave meaningful room for a learned model to improve on this strong domain-specific reference.

## 8. Linear Models

Linear Regression tests whether the engineered predictors add value through a simple additive relationship. Ridge Regression uses the same structure with an $L_2$ penalty, providing a first indication of whether regularization helps when predictors are correlated or expand through categorical encoding. Both models are fitted on training data and evaluated only on validation data.

### Preprocessing

Feature types are determined from their stored data types. Numeric and boolean predictors are median-imputed and standardized; the history-derived missing values are therefore retained rather than causing rows to be dropped. Categorical predictors are most-frequent-imputed and one-hot encoded, with unseen validation categories ignored. Standardization is especially important for Ridge because its penalty acts on coefficient magnitudes and should not depend on the original measurement units.

In [ ]:
boolean_model_features = [
    column
    for column in all_model_features
    if pd.api.types.is_bool_dtype(X_train[column])
]
numeric_model_features = [
    column
    for column in all_model_features
    if pd.api.types.is_numeric_dtype(X_train[column])
    and column not in boolean_model_features
]
categorical_model_features = [
    column
    for column in all_model_features
    if column not in numeric_model_features + boolean_model_features
]

pd.DataFrame(
    {
        "feature_count": [
            len(numeric_model_features),
            len(categorical_model_features),
            len(boolean_model_features),
        ],
        "features_with_missing_values": [
            X_train[numeric_model_features].isna().any().sum(),
            X_train[categorical_model_features].isna().any().sum(),
            X_train[boolean_model_features].isna().any().sum(),
        ],
    },
    index=["numeric", "categorical", "boolean"],
)

In [ ]:
numeric_and_boolean_features = (
    numeric_model_features + boolean_model_features
)
numeric_preprocessing = Pipeline(
    steps=[
        ("imputation", SimpleImputer(strategy="median")),
        ("scaling", StandardScaler()),
    ]
)
categorical_preprocessing = Pipeline(
    steps=[
        ("imputation", SimpleImputer(strategy="most_frequent")),
        ("encoding", OneHotEncoder(handle_unknown="ignore")),
    ]
)
preprocessing = ColumnTransformer(
    transformers=[
        ("numeric", numeric_preprocessing, numeric_and_boolean_features),
        ("categorical", categorical_preprocessing, categorical_model_features),
    ]
)

### Linear Regression

The ordinary linear model estimates an additive relationship between the preprocessed predictors and final attendance without regularization.

In [ ]:
linear_model = Pipeline(
    steps=[
        ("preprocessing", clone(preprocessing)),
        ("model", LinearRegression()),
    ]
)
linear_model.fit(X_train, y_train)
linear_validation_prediction = linear_model.predict(X_validation)

### Ridge Regression

Ridge uses the same preprocessed inputs with moderate regularization ($\alpha = 1.0$). The value is fixed for this first comparison rather than tuned against validation performance.

In [ ]:
ridge_model = Pipeline(
    steps=[
        ("preprocessing", clone(preprocessing)),
        ("model", Ridge(alpha=1.0)),
    ]
)
ridge_model.fit(X_train, y_train)
ridge_validation_prediction = ridge_model.predict(X_validation)

### Validation Performance

The learned models are compared with both validation benchmarks using the same metrics and sign convention.

In [ ]:
linear_model_results = pd.DataFrame(
    [
        {
            "model": "Linear regression",
            **evaluate_predictions(y_validation, linear_validation_prediction),
        },
        {
            "model": "Ridge regression",
            **evaluate_predictions(y_validation, ridge_validation_prediction),
        },
    ]
).set_index("model")
validation_results = pd.concat([benchmark_results, linear_model_results])

validation_results.round(2)

### Interpretation

Linear Regression improves on the training-mean baseline and also modestly outperforms the current-booking benchmark, suggesting that the engineered features contain additional signal that a simple additive relationship can capture. Ridge improves further on both Linear Regression and the booking-count baseline, indicating that regularization is useful for this correlated and one-hot-expanded feature set at $\alpha = 1.0$. Both learned models have negative mean error and therefore tend to underpredict, although this bias is smaller for Ridge. These results make Ridge the stronger linear reference for the subsequent nonlinear model comparison.

## 9. Random Forest Regression

Random Forest provides the first nonlinear reference. Its trees can represent thresholds and interactions between booking state, capacity, historical behavior, and class context without specifying those relationships manually. The model configuration is fixed rather than tuned, and evaluation remains limited to the validation period.

### Preprocessing

Tree splits depend on feature ordering and thresholds rather than coefficient magnitudes, so numerical scaling is unnecessary. Median imputation is retained for numeric and boolean predictors, while categorical predictors use the same imputation and one-hot encoding as the linear models.

In [ ]:
def make_tree_preprocessing(feature_columns):
    numeric_columns = [
        column
        for column in numeric_and_boolean_features
        if column in feature_columns
    ]
    categorical_columns = [
        column
        for column in categorical_model_features
        if column in feature_columns
    ]
    return ColumnTransformer(
        transformers=[
            ("numeric", SimpleImputer(strategy="median"), numeric_columns),
            (
                "categorical",
                clone(categorical_preprocessing),
                categorical_columns,
            ),
        ]
    )


tree_preprocessing = make_tree_preprocessing(all_model_features)

### Fit the Model

In [ ]:
random_forest_model = Pipeline(
    steps=[
        ("preprocessing", tree_preprocessing),
        (
            "model",
            RandomForestRegressor(
                n_estimators=300,
                random_state=42,
                n_jobs=-1,
            ),
        ),
    ]
)
random_forest_model.fit(X_train, y_train)
random_forest_validation_prediction = random_forest_model.predict(X_validation)

### Validation Performance

The Random Forest is added to the same validation comparison used for the benchmarks and linear models.

In [ ]:
random_forest_results = pd.DataFrame(
    [
        {
            "model": "Random forest",
            **evaluate_predictions(
                y_validation, random_forest_validation_prediction
            ),
        }
    ]
).set_index("model")
validation_results = pd.concat([validation_results, random_forest_results])

validation_results.round(2)

### Interpretation

Random Forest improves both MAE and RMSE relative to Ridge and has a smaller negative mean error, although it still slightly underpredicts final attendance. This improvement suggests that nonlinear structure and interactions among the engineered predictors may add useful signal beyond the additive linear models, but it does not identify which relationships are responsible. The result supports retaining Random Forest as the nonlinear reference for the subsequent feature-group analysis.

## 10. Gradient Boosting Regression

Random Forest builds trees largely independently and averages their predictions. Gradient Boosting instead builds trees sequentially, with each new tree reducing errors left by the current ensemble. Both approaches can represent nonlinear relationships and interactions; this final model-family comparison tests whether sequential error correction improves validation performance.

### Fit the Model

The existing tree preprocessing is cloned so that all imputation and encoding is fitted independently on training data. Estimator defaults are retained, with only the random seed fixed for reproducibility.

In [ ]:
gradient_boosting_model = Pipeline(
    steps=[
        ("preprocessing", clone(tree_preprocessing)),
        ("model", GradientBoostingRegressor(random_state=42)),
    ]
)
gradient_boosting_model.fit(X_train, y_train)
gradient_boosting_validation_prediction = gradient_boosting_model.predict(
    X_validation
)

### Validation Performance

Gradient Boosting completes the shared validation comparison for this model-selection stage.

In [ ]:
gradient_boosting_results = pd.DataFrame(
    [
        {
            "model": "Gradient boosting",
            **evaluate_predictions(
                y_validation, gradient_boosting_validation_prediction
            ),
        }
    ]
).set_index("model")
validation_results = pd.concat([validation_results, gradient_boosting_results])

validation_results.round(2)

### Interpretation

Gradient Boosting performs very similarly to Random Forest but is marginally worse on both MAE and RMSE, so sequential error correction does not improve on the bagged-tree model in this untuned comparison. Both nonlinear models outperform Ridge and the current-booking benchmark, which suggests that nonlinear structure is useful, while their negative mean errors indicate only slight underprediction. Random Forest therefore remains the strongest current candidate, although the small difference between the tree models should not be overstated. 

## 11. Cumulative Feature-Group Analysis

The model family and settings are now held fixed while increasingly rich information is added in the order used during feature engineering. This is a cumulative forward analysis, not a full-model ablation: an incremental change can depend on all groups included before it and should not be interpreted as a unique or causal contribution. Random Forest is used throughout because it is the strongest validation model in the preceding comparison.

### Cumulative Feature Sets

Fresh preprocessing and a fresh Random Forest are fitted on the same training rows for every cumulative subset. Validation rows remain unchanged, and missing values are handled inside each training-fitted pipeline.

In [ ]:
feature_group_order = [
    ("baseline", "Baseline"),
    ("historical", "+ Historical"),
    ("member_history", "+ Member History"),
    ("member_reliability", "+ Member Reliability"),
    ("booking_dynamics", "+ Booking Dynamics"),
    ("class_context", "+ Class Context"),
]
cumulative_features = []
feature_group_results = []

for group_name, feature_set_label in feature_group_order:
    cumulative_features.extend(feature_groups[group_name])
    selected_features = cumulative_features.copy()
    feature_group_model = Pipeline(
        steps=[
            (
                "preprocessing",
                make_tree_preprocessing(selected_features),
            ),
            (
                "model",
                RandomForestRegressor(
                    n_estimators=300,
                    random_state=42,
                    n_jobs=-1,
                ),
            ),
        ]
    )
    feature_group_model.fit(X_train[selected_features], y_train)
    prediction = feature_group_model.predict(X_validation[selected_features])
    feature_group_results.append(
        {
            "feature_set": feature_set_label,
            "n_features": len(selected_features),
            **evaluate_predictions(y_validation, prediction),
        }
    )

cumulative_feature_results = pd.DataFrame(feature_group_results).set_index(
    "feature_set"
)
cumulative_feature_results["MAE improvement vs previous"] = (
    cumulative_feature_results["MAE"].shift(1)
    - cumulative_feature_results["MAE"]
)

### Validation Performance

Positive values in `MAE improvement vs previous` indicate that adding the next group reduced validation error. `n_features` counts original selected columns before one-hot encoding.

In [ ]:
cumulative_feature_results.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(
    cumulative_feature_results.index,
    cumulative_feature_results["MAE"],
    color="#315f72",
    marker="o",
)
ax.set(xlabel="Cumulative feature set", ylabel="Validation MAE")
ax.tick_params(axis="x", rotation=25)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()

### Interpretation

The baseline group alone already outperforms the current-booking benchmark, indicating that current class and booking-state variables provide substantial predictive information. Adding Historical features improves validation MAE, and Member History provides a smaller further improvement. Member Reliability then worsens performance slightly after those earlier groups, while Booking Dynamics reverses that loss and produces the best cumulative validation result. Adding Class Context slightly worsens performance again. These differences are modest and order-dependent: they suggest incremental signal, overlap, or noise under the fixed Random Forest rather than unique contributions from individual groups.

## 12. Baseline + One Feature Group

The compact Baseline already represents the current class and booking state. Because the cumulative analysis found only small gains from adding many engineered predictors, each additional information source is now evaluated from the same Baseline starting point. This supports a parsimonious choice: extra complexity should provide a meaningful validation improvement. Small differences do not imply that a group contains no information; its signal may overlap with the Baseline or be difficult to distinguish from validation noise.

### Validation Performance

Each candidate uses the same training and validation prediction instances, fresh training-fitted preprocessing, and the fixed Random Forest. Unlike cumulative forward addition, every additional group is compared directly on top of Baseline.

In [ ]:
additional_feature_groups = [
    ("historical", "Baseline + Historical"),
    ("member_history", "Baseline + Member History"),
    ("member_reliability", "Baseline + Member Reliability"),
    ("booking_dynamics", "Baseline + Booking Dynamics"),
    ("class_context", "Baseline + Class Context"),
]

candidate_feature_sets = {
    "Baseline": feature_groups["baseline"].copy(),
}
for group_name, feature_set_label in additional_feature_groups:
    candidate_feature_sets[feature_set_label] = list(
        dict.fromkeys(
            feature_groups["baseline"] + feature_groups[group_name]
        )
    )

compact_feature_rows = []
for feature_set_label, selected_features in candidate_feature_sets.items():
    candidate_model = Pipeline(
        steps=[
            ("preprocessing", make_tree_preprocessing(selected_features)),
            (
                "model",
                RandomForestRegressor(
                    n_estimators=300,
                    random_state=42,
                    n_jobs=-1,
                ),
            ),
        ]
    )
    candidate_model.fit(X_train[selected_features], y_train)
    prediction = candidate_model.predict(
        X_validation[selected_features]
    )
    compact_feature_rows.append(
        {
            "feature_set": feature_set_label,
            "n_features": len(selected_features),
            **evaluate_predictions(y_validation, prediction),
        }
    )

compact_feature_results = pd.DataFrame(compact_feature_rows).set_index(
    "feature_set"
)
baseline_mae = compact_feature_results.at["Baseline", "MAE"]
compact_feature_results["MAE improvement vs Baseline"] = (
    baseline_mae - compact_feature_results["MAE"]
)

full_model_metrics = random_forest_results.loc["Random forest"]
compact_feature_results.loc["All feature groups (reference)"] = {
    "n_features": len(all_model_features),
    "MAE": full_model_metrics["MAE"],
    "RMSE": full_model_metrics["RMSE"],
    "mean_error": full_model_metrics["mean_error"],
    "MAE improvement vs Baseline": (
        baseline_mae - full_model_metrics["MAE"]
    ),
}

Positive `MAE improvement vs Baseline` means that the additional group reduced validation MAE. `n_features` counts selected columns before one-hot encoding; the final row reuses the existing full-model result as a reference rather than as another compact candidate.

In [ ]:
compact_feature_results.round(3)

In [ ]:
additional_group_improvements = (
    compact_feature_results["MAE improvement vs Baseline"]
    .drop(index=["Baseline", "All feature groups (reference)"])
    .sort_values()
)
bar_colors = [
    "#2f6f5e" if value > 0 else "#a44a3f"
    for value in additional_group_improvements
]

fig, ax = plt.subplots(figsize=(8, 4.5))
additional_group_improvements.plot.barh(ax=ax, color=bar_colors)
ax.axvline(0, color="#333333", linewidth=1)
ax.set(xlabel="MAE improvement vs Baseline", ylabel="")
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()

### Interpretation

The compact Baseline achieves a validation MAE of 1.499 with 14 features. Adding Member Reliability provides the largest improvement, reducing MAE by 0.063 to 1.436 with 26 features. This gain is small in absolute attendance units, but the compact candidate also outperforms the 83-feature reference model, whose MAE is 1.471. Historical improves MAE by 0.023, while Class Context changes it by only 0.005. Member History and Booking Dynamics worsen MAE by 0.008 and 0.024 when each is added directly to Baseline. The contrast with the cumulative analysis shows that a group's usefulness depends on the information already available: Member Reliability helps from the Baseline starting point but not after Historical and Member History, whereas Booking Dynamics helps later in the cumulative sequence but not on its own. Given its stronger validation result and substantially smaller representation, Baseline + Member Reliability is a reasonable candidate for subsequent tuning and interpretation. These differences come from a limited validation sample and do not prove that excluded groups lack information; their signal may be redundant or interaction-dependent. The test set remains untouched.

## 13. Random Forest Hyperparameter Tuning

Hyperparameter selection can itself overfit: repeatedly comparing many configurations on one validation period may reward random peculiarities of that period, particularly when observed MAE differences are only a few hundredths of an attendee. Tuning therefore uses only the existing training period. The outer validation period remains outside this search, and the test period remains completely untouched.

### Why Temporal Cross-Validation?

The deployment question is chronological: learn from past classes and predict later classes. Three expanding-window folds preserve this direction without shuffling. Every fold is formed from complete physical classes through `class_start`, so prediction horizons from the same class cannot cross fold boundaries. Three folds balance repeated future-period evaluation with meaningful training samples in this relatively small dataset. Learned preprocessing is refitted independently inside every fold.

In [ ]:
selected_compact_features = list(
    dict.fromkeys(
        feature_groups["baseline"]
        + feature_groups["member_reliability"]
    )
)

training_class_starts = (
    train["class_start"]
    .drop_duplicates()
    .sort_values()
    .reset_index(drop=True)
)
number_of_class_starts = len(training_class_starts)
cv_validation_boundaries = [
    training_class_starts.iloc[number_of_class_starts // 4],
    training_class_starts.iloc[number_of_class_starts // 2],
    training_class_starts.iloc[3 * number_of_class_starts // 4],
    training_end,
]

temporal_cv_folds = []
cv_fold_summary_rows = []
for fold_number, (cv_validation_start, cv_validation_end) in enumerate(
    zip(cv_validation_boundaries[:-1], cv_validation_boundaries[1:]),
    start=1,
):
    cv_train = train.loc[train["class_start"] < cv_validation_start]
    cv_validation = train.loc[
        train["class_start"].ge(cv_validation_start)
        & train["class_start"].lt(cv_validation_end)
    ]
    temporal_cv_folds.append((cv_train.index, cv_validation.index))
    cv_fold_summary_rows.append(
        {
            "fold": fold_number,
            "training_start": cv_train["class_start"].min(),
            "training_end": cv_train["class_start"].max(),
            "training_classes": len(
                cv_train[physical_class_identifier_columns].drop_duplicates()
            ),
            "validation_start": cv_validation["class_start"].min(),
            "validation_end": cv_validation["class_start"].max(),
            "validation_classes": len(
                cv_validation[
                    physical_class_identifier_columns
                ].drop_duplicates()
            ),
        }
    )

cv_fold_summary = pd.DataFrame(cv_fold_summary_rows).set_index("fold")
cv_fold_summary

### Coarse Search

The coarse search changes only two regularization controls. `min_samples_leaf` progresses from flexible leaves (`1`) through light (`2`), moderate (`4`), and stronger smoothing (`8`). `max_features` compares all predictors (`1.0`), approximately half (`0.5`), and a substantially smaller random subset (`"sqrt"`), which can reduce correlation among trees. `max_depth=None`, 300 trees, and the random seed remain fixed. Twelve deliberately chosen configurations limit selection noise; more trees are not tuned because 300 already provide stable averaging rather than changing the central complexity trade-off.

In [ ]:
def evaluate_random_forest_cv(max_features, min_samples_leaf, max_depth):
    """Evaluate one Random Forest configuration across temporal folds."""
    fold_mae = []
    for cv_train_index, cv_validation_index in temporal_cv_folds:
        cv_train = train.loc[cv_train_index]
        cv_validation = train.loc[cv_validation_index]
        cv_model = Pipeline(
            steps=[
                (
                    "preprocessing",
                    make_tree_preprocessing(selected_compact_features),
                ),
                (
                    "model",
                    RandomForestRegressor(
                        n_estimators=300,
                        max_features=max_features,
                        min_samples_leaf=min_samples_leaf,
                        max_depth=max_depth,
                        random_state=42,
                        n_jobs=-1,
                    ),
                ),
            ]
        )
        cv_model.fit(
            cv_train[selected_compact_features],
            cv_train[target_column],
        )
        cv_prediction = cv_model.predict(
            cv_validation[selected_compact_features]
        )
        fold_mae.append(
            mean_absolute_error(cv_validation[target_column], cv_prediction)
        )
    return fold_mae


coarse_search_rows = []
for max_features_value in [1.0, 0.5, "sqrt"]:
    for min_samples_leaf_value in [1, 2, 4, 8]:
        fold_mae = evaluate_random_forest_cv(
            max_features=max_features_value,
            min_samples_leaf=min_samples_leaf_value,
            max_depth=None,
        )
        coarse_search_rows.append(
            {
                "max_features": max_features_value,
                "min_samples_leaf": min_samples_leaf_value,
                "max_depth": "None",
                **{
                    f"fold_{fold_number}_MAE": mae
                    for fold_number, mae in enumerate(fold_mae, start=1)
                },
                "mean_CV_MAE": pd.Series(fold_mae).mean(),
                "std_CV_MAE": pd.Series(fold_mae).std(),
            }
        )

coarse_search_results = (
    pd.DataFrame(coarse_search_rows)
    .sort_values(["mean_CV_MAE", "std_CV_MAE"])
    .reset_index(drop=True)
)

In [ ]:
coarse_search_results.round(3)

The lowest mean CV MAE is 1.367 for `max_features=0.5` and `min_samples_leaf=4`. The `sqrt` models with leaf sizes 1, 4, and 2 follow within 0.005 MAE; these differences are practically small. In particular, `sqrt` with leaf size 1 has a nearly identical mean of 1.369 and the lowest fold standard deviation of 0.014. Reducing `max_features` is generally helpful relative to using every predictor, while moderate leaf regularization helps the `0.5` and `1.0` regions; the strongest leaf smoothing does not improve the leading configurations.

### Targeted Depth Search

Depth is examined only for two coarse regions: the lowest-mean configuration (`max_features=0.5`, leaf size 4) and the nearly tied but more temporally stable configuration (`max_features="sqrt"`, leaf size 1). Comparing `None`, 6, 10, and 15 tests whether an explicit depth limit improves either trade-off without expanding the search across all coarse combinations.

In [ ]:
promising_regions = [
    {"max_features": 0.5, "min_samples_leaf": 4},
    {"max_features": "sqrt", "min_samples_leaf": 1},
]
max_depth_values = [None, 6, 10, 15]

targeted_search_rows = []
for region in promising_regions:
    for max_depth_value in max_depth_values:
        fold_mae = evaluate_random_forest_cv(
            max_features=region["max_features"],
            min_samples_leaf=region["min_samples_leaf"],
            max_depth=max_depth_value,
        )
        targeted_search_rows.append(
            {
                "max_features": region["max_features"],
                "min_samples_leaf": region["min_samples_leaf"],
                "max_depth": (
                    "None" if max_depth_value is None else max_depth_value
                ),
                **{
                    f"fold_{fold_number}_MAE": mae
                    for fold_number, mae in enumerate(fold_mae, start=1)
                },
                "mean_CV_MAE": pd.Series(fold_mae).mean(),
                "std_CV_MAE": pd.Series(fold_mae).std(),
            }
        )

targeted_search_results = (
    pd.DataFrame(targeted_search_rows)
    .sort_values(["mean_CV_MAE", "std_CV_MAE"])
    .reset_index(drop=True)
)

In [ ]:
targeted_search_results.round(3)

The depth limits do not materially improve either region. `max_features=0.5`, leaf size 4, and depth 10 retains the lowest mean CV MAE of 1.367, but its fold standard deviation is 0.033. The unrestricted `sqrt` configuration with leaf size 1 is only about 0.002 MAE worse and substantially more stable across time, with a standard deviation of 0.014. Because this mean difference is negligible, the more stable `sqrt` configuration is selected without an additional depth constraint.

### CV-Selected Configuration

The CV-selected Random Forest uses `max_features="sqrt"`, `min_samples_leaf=1`, and `max_depth=None`, with 300 trees and the fixed random seed. This choice prioritizes temporal stability over a roughly two-thousandths advantage in mean CV MAE. The compact Baseline + Member Reliability feature set remains fixed.

In [ ]:
cv_selected_random_forest_parameters = {
    "n_estimators": 300,
    "max_features": "sqrt",
    "min_samples_leaf": 1,
    "max_depth": None,
    "random_state": 42,
    "n_jobs": -1,
}
cv_selected_random_forest_parameters

### Outer Validation Check

The CV-selected configuration is now fitted once on the complete outer training period and compared with the previous untuned compact Random Forest on the outer validation period. This is a final model-development check, not a completely independent estimate: the same validation period was already used for model-family and feature-set selection. No further tuning decisions are made from small differences here; the test period remains the untouched final evaluation sample.

In [ ]:
tuned_compact_model = Pipeline(
    steps=[
        (
            "preprocessing",
            make_tree_preprocessing(selected_compact_features),
        ),
        (
            "model",
            RandomForestRegressor(**cv_selected_random_forest_parameters),
        ),
    ]
)
tuned_compact_model.fit(
    X_train[selected_compact_features],
    y_train,
)
tuned_compact_validation_prediction = tuned_compact_model.predict(
    X_validation[selected_compact_features]
)

untuned_compact_metrics = compact_feature_results.loc[
    "Baseline + Member Reliability",
    ["MAE", "RMSE", "mean_error"],
]
outer_validation_results = pd.DataFrame(
    [
        {
            "configuration": "Untuned compact Random Forest",
            **untuned_compact_metrics.to_dict(),
        },
        {
            "configuration": "Tuned compact Random Forest",
            **evaluate_predictions(
                y_validation,
                tuned_compact_validation_prediction,
            ),
        },
    ]
).set_index("configuration")
outer_validation_results["MAE improvement vs untuned"] = (
    outer_validation_results.at["Untuned compact Random Forest", "MAE"]
    - outer_validation_results["MAE"]
)
outer_validation_results.round(3)

### Interpretation

Temporal cross-validation selected the alternative Random Forest configuration, but it did not improve performance on the later outer validation period. Therefore, the original default Random Forest configuration is retained for subsequent model interpretation and final evaluation.

## 14. Interpreting the Compact Random Forest

The retained default compact Random Forest is interpreted only after fixing its feature set and configuration. Its 26 raw predictors are easier to inspect and communicate than all 83 engineered features, with less redundancy across feature groups. Permutation importance measures how much validation MAE worsens when one predictor is shuffled while the fitted model and all other columns remain unchanged. Applying it to the complete pipeline preserves the original feature level rather than ranking one-hot encoded columns.

Permutation importance describes predictive reliance, not causality or direction. Correlated predictors can substitute for one another, so low importance does not prove that a feature or concept is uninformative. The repeated-permutation standard deviation provides a descriptive measure of uncertainty, and small differences are not treated as significant. The frozen features and retained default model will not be changed from this outer-validation interpretation.

### Permutation Importance

The final compact pipeline is refitted with the retained default Random Forest settings on the complete outer training period. With `scoring="neg_mean_absolute_error"`, scikit-learn's importance difference equals the increase in MAE after permutation: positive values mean that shuffling the feature made validation predictions worse.

In [ ]:
final_compact_model = Pipeline(
    steps=[
        (
            "preprocessing",
            make_tree_preprocessing(selected_compact_features),
        ),
        (
            "model",
            RandomForestRegressor(
                n_estimators=300,
                max_features=1.0,
                min_samples_leaf=1,
                max_depth=None,
                random_state=42,
                n_jobs=-1,
            ),
        ),
    ]
)
final_compact_model.fit(
    X_train[selected_compact_features],
    y_train,
)

permutation_result = permutation_importance(
    final_compact_model,
    X_validation[selected_compact_features],
    y_validation,
    scoring="neg_mean_absolute_error",
    n_repeats=20,
    random_state=42,
    n_jobs=-1,
)

compact_feature_group = {
    feature: "Baseline" for feature in feature_groups["baseline"]
}
compact_feature_group.update(
    {
        feature: "Member Reliability"
        for feature in feature_groups["member_reliability"]
    }
)

permutation_importance_results = (
    pd.DataFrame(
        {
            "feature": selected_compact_features,
            "feature_group": [
                compact_feature_group[feature]
                for feature in selected_compact_features
            ],
            "mean_permutation_importance": (
                permutation_result.importances_mean
            ),
            "std_permutation_importance": (
                permutation_result.importances_std
            ),
        }
    )
    .sort_values("mean_permutation_importance", ascending=False)
    .reset_index(drop=True)
)

In [ ]:
permutation_importance_results.round(3)

In [ ]:
importance_plot_data = (
    permutation_importance_results.head(15)
    .sort_values("mean_permutation_importance")
)

fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(
    importance_plot_data["feature"],
    importance_plot_data["mean_permutation_importance"],
    xerr=importance_plot_data["std_permutation_importance"],
    color="#315f72",
    alpha=0.9,
    capsize=3,
)
ax.axvline(0, color="#333333", linewidth=1)
ax.set(xlabel="Increase in validation MAE after permutation", ylabel="")
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()

### Interpretation

Current booking-state information dominates the fitted model. attendance_count has by far the largest permutation importance, followed by occupancy_rate, while weekday and course contribute smaller additional signal. This supports the earlier finding that the compact Baseline captures most of the predictive information. Because attendance, occupancy, available spots, and capacity are correlated, their individual permutation importances should not be interpreted as the total importance of the underlying booking-state concept.

Member Reliability nevertheless provides visible additional signal. Among these features, the long-term member_show_up_rate_mean_365d is the strongest predictor, while several measures of available reliability history also contribute. Shorter-window show-up-rate estimates and reliability-history shares show comparatively small or uncertain importance. Negative permutation importance for an individual feature is treated as validation-sample variation rather than evidence that the feature should be removed. Permutation importance measures predictive reliance, not the direction of an effect. 

## 15. Final Test Evaluation

Model-family selection, feature-set selection, hyperparameter investigation, and interpretation are now complete. The frozen model uses the Baseline + Member Reliability features and the retained default Random Forest configuration: 300 trees, `max_features=1.0`, `min_samples_leaf=1`, `max_depth=None`, and random seed 42. These choices will not change after observing test performance.

The test period was held out from all preceding model-development decisions. Train and validation are now combined so that the final model learns from all classes preceding the test period. The test set is used once for final performance estimation, not for further optimization.

### Final Refit

A fresh preprocessing pipeline and the frozen Random Forest are fitted only on the combined train and validation periods. No learned preprocessing or model component is fitted on test data.

In [ ]:
final_features = selected_compact_features.copy()
final_random_forest_parameters = {
    "n_estimators": 300,
    "max_features": 1.0,
    "min_samples_leaf": 1,
    "max_depth": None,
    "random_state": 42,
    "n_jobs": -1,
}

final_training = pd.concat([train, validation], axis=0)
test = features.loc[features["split"] == "test"]

X_final_train = final_training[final_features]
y_final_train = final_training[target_column]
X_test_final = test[final_features]
y_test_final = test[target_column]

final_model = Pipeline(
    steps=[
        ("preprocessing", make_tree_preprocessing(final_features)),
        (
            "model",
            RandomForestRegressor(**final_random_forest_parameters),
        ),
    ]
)
final_model.fit(X_final_train, y_final_train)
final_test_prediction = final_model.predict(X_test_final)

FINAL_TEST_PREDICTIONS_PATH = (
    Path("..") / "data" / "processed" / "final_test_predictions.parquet"
)
prediction_artifact_columns = [
    "studio",
    "course",
    "class_start",
    "prediction_horizon",
    "weekday",
    "capacity",
    "occupancy_rate",
    "snapshot_age_hours",
    "final_attendance_count",
    "attendance_count",
    "members_with_reliability_history_count_365d",
    "member_show_up_rate_mean_365d",
]
final_test_predictions = test[prediction_artifact_columns].copy()
final_test_predictions["final_model_prediction"] = final_test_prediction
final_test_predictions.to_parquet(
    FINAL_TEST_PREDICTIONS_PATH,
    index=False,
)

### Test Performance

The compact Random Forest test result is shown alongside its earlier outer-validation result and the current-booking-count heuristic on the same test period. Validation and test cover different future periods, so their difference provides temporal context rather than a controlled model comparison. The test benchmark is descriptive and does not reopen model selection.

In [ ]:
final_test_metrics = evaluate_predictions(
    y_test_final,
    final_test_prediction,
)
current_booking_test_metrics = evaluate_predictions(
    y_test_final,
    test[current_booking_feature],
)

final_evaluation_results = pd.DataFrame(
    [
        {
            "result": "Compact RF - outer validation",
            "prediction_instances": len(validation),
            "physical_classes": len(
                validation[
                    physical_class_identifier_columns
                ].drop_duplicates()
            ),
            **untuned_compact_metrics.to_dict(),
        },
        {
            "result": "Compact RF - final test",
            "prediction_instances": len(test),
            "physical_classes": len(
                test[physical_class_identifier_columns].drop_duplicates()
            ),
            **final_test_metrics,
        },
        {
            "result": "Current booking count - test benchmark",
            "prediction_instances": len(test),
            "physical_classes": len(
                test[physical_class_identifier_columns].drop_duplicates()
            ),
            **current_booking_test_metrics,
        },
    ]
).set_index("result")
final_evaluation_results.round(3)

### Final Interpretation

The frozen compact Random Forest achieves a final test MAE of 1.546 attendees and an RMSE of 1.828. Test MAE is somewhat higher than the earlier outer-validation MAE of 1.436, while RMSE remains nearly unchanged. The mean error of -0.095 indicates little systematic bias on the test period. Overall, the model therefore generalizes reasonably well to the later held-out classes, although average absolute error is somewhat higher than during model development.

The current-booking-count heuristic is already a strong benchmark on the test period, with an MAE of 1.680 and an RMSE of 2.213. The final model reduces MAE by only 0.134 attendees, so the improvement in typical prediction error is modest. The larger reduction in RMSE, however, suggests that the model is more successful at avoiding some of the larger prediction errors that occur when relying on the current booking count alone.

Taken together, the results show that the current booking state contains most of the predictive information available for final attendance, while Member Reliability provides a smaller additional signal. The compact 26-feature model performed better than the full 83-feature representation during validation, additional Random Forest tuning did not provide a convincing improvement, and permutation importance likewise showed that booking-state variables dominate the fitted model. The final test result supports a modest but genuine predictive benefit over the simple domain heuristic without requiring the much larger feature set. Model development is closed at this point; no decisions are revised based on the test result.